# EEG Sleep Stage Classification — Fixed Notebook
**Dataset:** BOAS Sleep (sub-1) — PSG EDF + TSV annotations  
**Models:** Random Forest, Gradient Boosting, XGBoost, LightGBM  
**Fix summary:** SMOTE inside CV loop (no leakage), correct imports, annotation loading, feature extraction, consistent classes

---
**Run all cells in order from top to bottom.**

## CELL 1 — Mount Google Drive

In [ ]:
# FIX 11: Mount drive ONCE only (removed duplicate mount at end of notebook)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## CELL 2 — Install Libraries
**FIX 2 & 9:** `!pip install` runs BEFORE any imports. `scikit-learn` imported as `sklearn`.

In [ ]:
# FIX 2: Install ALL libraries FIRST before any import statement
!pip install mne numpy pandas scipy scikit-learn imbalanced-learn xgboost lightgbm matplotlib seaborn -q
print('All libraries installed.')

## CELL 3 — Import Libraries

In [ ]:
# FIX 9: scikit-learn is imported as 'sklearn', never 'scikit_learn'
import mne
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import skew, kurtosis
from scipy.signal import welch

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score
)
from sklearn.preprocessing import StandardScaler

# FIX 9: imblearn, xgboost, lightgbm — correct import names
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print('All libraries imported successfully.')

## CELL 4 — Configuration

In [ ]:
# ── Update this path to match your Google Drive folder ────────
EDF_DIR = '/content/drive/MyDrive/Datasets/BOAS_Sleep/sub-1/eeg/'

# EEG channels to use (F3, F4, C3, C4 equivalents in PSG file)
TARGET_EEG_CHANNELS = ['PSG_F3', 'PSG_F4', 'PSG_C3', 'PSG_C4']

# Epoch length in seconds (standard for sleep staging)
EPOCH_DURATION = 30.0

# Band-pass filter range
L_FREQ, H_FREQ = 0.5, 30.0

# Artifact rejection threshold (microvolts)
REJECT_THRESH = dict(eeg=200e-6)

# Frequency bands for PSD features
FREQ_BANDS = {
    'delta': (0.5, 4),
    'theta': (4,   8),
    'alpha': (8,  13),
    'beta':  (13, 30),
}

# Cross-validation folds
N_SPLITS = 5
RANDOM_STATE = 42

print('Configuration set.')
print(f'  EDF directory : {EDF_DIR}')
print(f'  EEG channels  : {TARGET_EEG_CHANNELS}')
print(f'  Epoch duration: {EPOCH_DURATION}s')

## CELL 5 — List EDF Files
**FIX 4:** `all_items` defined HERE so it is available for all later cells.

In [ ]:
# FIX 4: all_items defined before it is ever used
all_items = os.listdir(EDF_DIR)

edf_files = [
    os.path.join(EDF_DIR, f)
    for f in all_items
    if f.endswith('.edf')
]

print(f'Found {len(edf_files)} EDF file(s):')
for f in edf_files:
    print(' ', f)

## CELL 6 — Load EDF Files and Annotations
**FIX 5:** Use `raw.filenames[0]` not `raw.info['filename']`.  
**FIX 6:** TSV annotations loaded via pandas with correct column mapping.

In [ ]:
def load_tsv_annotations(tsv_path, raw_sfreq):
    """
    FIX 6: Load a BIDS-style *_events.tsv file into an mne.Annotations object.
    Tries multiple column name variants so it works even if columns differ.
    Returns None if the file cannot be loaded.
    """
    try:
        df = pd.read_csv(tsv_path, sep='\t')
        print(f'    TSV columns found: {df.columns.tolist()}')

        # Try common column names for onset
        onset_col = next(
            (c for c in df.columns if c.lower() in ['onset', 'start', 'onset_time']),
            None
        )
        # Try common column names for duration
        dur_col = next(
            (c for c in df.columns if c.lower() in ['duration', 'dur', 'length']),
            None
        )
        # Try common column names for description / sleep stage
        desc_col = next(
            (c for c in df.columns
             if c.lower() in ['description', 'stage', 'stage_hum', 'label',
                               'annotation', 'value', 'trial_type']),
            None
        )

        if onset_col is None or desc_col is None:
            print(f'    Could not find onset or description column in {tsv_path}')
            print(f'    Available columns: {df.columns.tolist()}')
            return None

        # If duration column missing, use epoch length as default
        if dur_col is None:
            print(f'    Duration column not found — using {EPOCH_DURATION}s default')
            df['_dur'] = EPOCH_DURATION
            dur_col = '_dur'

        # Drop rows with NaN in critical columns
        df = df.dropna(subset=[onset_col, desc_col])

        annotations = mne.Annotations(
            onset       = df[onset_col].values.astype(float),
            duration    = df[dur_col].values.astype(float),
            description = df[desc_col].astype(str).values
        )
        print(f'    Loaded {len(annotations)} annotations from TSV')
        return annotations

    except Exception as e:
        print(f'    TSV load failed: {e}')
        return None


# ── Load all EDF files ────────────────────────────────────────
raw_data_objects = []

for edf_path in edf_files:
    fname = os.path.basename(edf_path)
    print(f'\n--- Loading: {fname} ---')

    try:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

        # FIX 5: raw.filenames[0] is correct, NOT raw.info['filename']
        print(f'  Channels    : {raw.info["ch_names"]}')
        print(f'  Sampling Hz : {raw.info["sfreq"]}')
        print(f'  Shape       : {raw.get_data().shape}')

        # Build expected TSV path from EDF name
        base  = fname.replace('_eeg.edf', '_events.tsv')
        tsv   = os.path.join(EDF_DIR, base)

        if os.path.exists(tsv):
            print(f'  Annotation file found: {base}')
            annots = load_tsv_annotations(tsv, raw.info['sfreq'])
            if annots is not None:
                raw.set_annotations(annots)
                print(f'  Annotations set: {len(raw.annotations)}')
            else:
                print('  No valid annotations — file will be skipped in feature extraction')
        else:
            print(f'  No TSV annotation file found at: {tsv}')

        raw_data_objects.append(raw)

    except Exception as e:
        print(f'  ERROR loading {fname}: {e}')

print(f'\nLoaded {len(raw_data_objects)} raw object(s) total.')

## CELL 7 — Verify Loaded Datasets

In [ ]:
print(f'--- Verification: {len(raw_data_objects)} dataset(s) ---')

for i, raw in enumerate(raw_data_objects):
    # FIX 5: use raw.filenames[0]
    print(f'\nDataset {i+1}: {os.path.basename(raw.filenames[0])}')
    print(f'  Channels  : {raw.info["ch_names"]}')
    print(f'  Sfreq     : {raw.info["sfreq"]} Hz')
    n_annots = len(raw.annotations) if raw.annotations else 0
    print(f'  Annotations: {n_annots}')
    if n_annots > 0:
        unique_desc = set(raw.annotations.description)
        print(f'  Unique stages: {unique_desc}')

## CELL 8 — Feature Extraction Helper Functions
**FIX 7:** Using PSD + statistical + Hjorth features (72 features) instead of 46,086 raw time-points.

In [ ]:
def hjorth_parameters(signal):
    """Compute Hjorth Activity, Mobility, Complexity for a 1D signal."""
    if np.std(signal) == 0:
        return 0.0, 0.0, 0.0

    activity  = float(np.var(signal))
    d1        = np.diff(signal)

    if np.std(d1) == 0 or activity == 0:
        return activity, 0.0, 0.0

    mobility  = float(np.sqrt(np.var(d1) / activity))
    d2        = np.diff(d1)
    mob_d1    = np.sqrt(np.var(d2) / np.var(d1)) if np.var(d1) > 0 else 0.0
    complexity = float(mob_d1 / mobility) if mobility > 0 else 0.0

    return activity, mobility, complexity


def band_power(signal, sfreq, band):
    """Compute mean PSD power in a frequency band using Welch's method."""
    freqs, psd = welch(signal, fs=sfreq, nperseg=min(256, len(signal)))
    idx  = np.logical_and(freqs >= band[0], freqs <= band[1])
    return float(np.mean(psd[idx])) if idx.any() else 0.0


def extract_features_from_epoch(epoch_data, sfreq):
    """
    FIX 7: Extract 14 features per channel:
      - 4 PSD band powers (delta, theta, alpha, beta)
      - 7 statistical (mean, std, min, max, var, skewness, kurtosis)
      - 3 Hjorth (activity, mobility, complexity)
    For 4 channels -> 56 features total.
    """
    features = []
    for ch_signal in epoch_data:
        # PSD features
        for band in FREQ_BANDS.values():
            features.append(band_power(ch_signal, sfreq, band))

        # Statistical features
        features += [
            float(np.mean(ch_signal)),
            float(np.std(ch_signal)),
            float(np.min(ch_signal)),
            float(np.max(ch_signal)),
            float(np.var(ch_signal)),
            float(skew(ch_signal)),
            float(kurtosis(ch_signal)),
        ]

        # Hjorth features
        a, m, c = hjorth_parameters(ch_signal)
        features += [a, m, c]

    return np.array(features, dtype=np.float32)


n_features = len(TARGET_EEG_CHANNELS) * (4 + 7 + 3)
print(f'Feature vector size: {n_features} ({len(TARGET_EEG_CHANNELS)} channels × 14 features)')

## CELL 9 — Preprocess and Extract Features
**FIX 8:** Sleep stage labels collected consistently. Missing stage 3 handled gracefully.

In [ ]:
# Sleep stage label mapping — update if your TSV uses different descriptions
STAGE_MAP = {
    # Numeric strings
    '0': 0, '1': 1, '2': 2, '3': 3, '4': 3, '5': 4,
    # AASM text labels
    'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'REM': 4, 'R': 4,
    # Some datasets use these
    'wake': 0, 'n1': 1, 'n2': 2, 'n3': 3, 'rem': 4,
    'Sleep stage W':  0,
    'Sleep stage 1':  1,
    'Sleep stage 2':  2,
    'Sleep stage 3':  3,
    'Sleep stage 4':  3,
    'Sleep stage R':  4,
    'Sleep stage REM':4,
}

all_features = []
all_labels   = []

for idx, raw in enumerate(raw_data_objects):
    fname = os.path.basename(raw.filenames[0])
    print(f'\n--- Processing {fname} ---')

    if len(raw.annotations) == 0:
        print('  Skipping: no annotations.')
        continue

    # ── Set channel types ────────────────────────────────────
    ch_type_map = {}
    for ch in raw.info['ch_names']:
        if ch in TARGET_EEG_CHANNELS:
            ch_type_map[ch] = 'eeg'
        elif 'EOG' in ch:
            ch_type_map[ch] = 'eog'
        elif 'EMG' in ch:
            ch_type_map[ch] = 'emg'
        else:
            ch_type_map[ch] = 'misc'
    raw.set_channel_types(ch_type_map)
    print(f'  Channel types set.')

    # ── Check target channels exist ───────────────────────────
    available_eeg = [c for c in TARGET_EEG_CHANNELS if c in raw.info['ch_names']]
    if len(available_eeg) == 0:
        print(f'  Skipping: none of {TARGET_EEG_CHANNELS} found.')
        continue
    print(f'  EEG channels available: {available_eeg}')

    # ── Band-pass filter ──────────────────────────────────────
    raw.filter(L_FREQ, H_FREQ, picks='eeg', verbose=False)
    print(f'  Band-pass {L_FREQ}–{H_FREQ} Hz applied.')

    # ── Map annotations to numeric labels ────────────────────
    onsets      = raw.annotations.onset
    descriptions= raw.annotations.description
    sfreq       = raw.info['sfreq']

    n_extracted = 0
    n_skipped   = 0

    for onset, desc in zip(onsets, descriptions):
        label = STAGE_MAP.get(str(desc).strip(), None)
        if label is None:
            n_skipped += 1
            continue

        start_s = onset
        stop_s  = onset + EPOCH_DURATION

        # Skip epochs that go beyond the recording
        if stop_s > raw.times[-1]:
            n_skipped += 1
            continue

        start_samp = int(start_s * sfreq)
        stop_samp  = int(stop_s  * sfreq)

        # Extract data for target EEG channels only
        picks = mne.pick_channels(raw.info['ch_names'], available_eeg)
        epoch_data = raw.get_data(picks=picks, start=start_samp, stop=stop_samp)

        # FIX 7: Extract compact features instead of raw samples
        feat = extract_features_from_epoch(epoch_data, sfreq)

        # Only add if no NaN/Inf in features
        if np.isfinite(feat).all():
            all_features.append(feat)
            all_labels.append(label)
            n_extracted += 1
        else:
            n_skipped += 1

    print(f'  Extracted: {n_extracted} epochs | Skipped: {n_skipped}')


all_features = np.array(all_features, dtype=np.float32)
all_labels   = np.array(all_labels,   dtype=np.int32)

print(f'\nFinal dataset:')
print(f'  Features shape : {all_features.shape}')
print(f'  Labels shape   : {all_labels.shape}')

## CELL 10 — Class Distribution and SMOTE
**FIX 8:** Labels consistent regardless of which stages appear in the data.

In [ ]:
# FIX 8: Check which stages are actually present
unique_labels, counts = np.unique(all_labels, return_counts=True)

STAGE_NAMES = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}

print('Original class distribution:')
for lbl, cnt in zip(unique_labels, counts):
    print(f'  Class {lbl} ({STAGE_NAMES.get(lbl,"?")}) : {cnt} samples')

# Bar chart
plt.figure(figsize=(7,4))
plt.bar([STAGE_NAMES.get(l,'?') for l in unique_labels], counts, color='steelblue', edgecolor='white')
plt.title('Sleep Stage Distribution (before SMOTE)')
plt.xlabel('Stage'); plt.ylabel('Count')
plt.tight_layout()
plt.savefig('stage_distribution_before.png', dpi=150)
plt.show()
print('Plot saved.')

## CELL 11 — Cross-Validation Setup
**FIX 3:** SMOTE will be applied INSIDE the CV loop on training folds only — not here on the full dataset.

In [ ]:
# FIX 3: Do NOT apply SMOTE here.
# SMOTE must be applied inside the cross-validation loop on each training fold only.
# Applying it to the full dataset before splitting causes data leakage and
# artificially inflates accuracy from ~80% to ~97%.

X = all_features.copy()
y = all_labels.copy()

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

scaler = StandardScaler()

print(f'Cross-validation ready:')
print(f'  Samples  : {X.shape[0]}')
print(f'  Features : {X.shape[1]}')
print(f'  Folds    : {N_SPLITS}')
print()
print('IMPORTANT: SMOTE will be applied INSIDE each fold on training data only.')
print('This prevents data leakage and gives honest accuracy estimates.')

## CELL 12 — Cross-Validation Runner
A reusable function that trains any model with proper SMOTE-inside-fold.

In [ ]:
def run_cv(model, X, y, kf, model_name='Model'):
    """
    Run stratified k-fold cross-validation.
    FIX 3: SMOTE applied ONLY to training fold — never to test fold.
    """
    acc_list, prec_list, rec_list, f1_list, kappa_list = [], [], [], [], []
    all_y_true, all_y_pred = [], []

    print(f'\n=== {model_name} — {N_SPLITS}-fold CV ===')

    for fold, (train_idx, test_idx) in enumerate(kf.split(X, y)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        # Scale using training statistics only
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)      # transform only — no fit on test

        # FIX 3: SMOTE on training fold only
        # Check that each class has at least k_neighbors+1 samples
        min_class_count = np.min(np.bincount(y_tr))
        k_neighbors = min(5, min_class_count - 1)

        if k_neighbors >= 1:
            smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)
            X_tr, y_tr = smote.fit_resample(X_tr, y_tr)
        else:
            print(f'  Fold {fold+1}: Skipping SMOTE (too few samples in a class)')

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        acc   = accuracy_score(y_te, y_pred)
        prec  = precision_score(y_te, y_pred, average='weighted', zero_division=0)
        rec   = recall_score(y_te, y_pred, average='weighted', zero_division=0)
        f1    = f1_score(y_te, y_pred, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(y_te, y_pred)

        acc_list.append(acc);   prec_list.append(prec)
        rec_list.append(rec);   f1_list.append(f1)
        kappa_list.append(kappa)
        all_y_true.extend(y_te)
        all_y_pred.extend(y_pred)

        print(f'  Fold {fold+1}: Acc={acc:.4f}  F1={f1:.4f}  Kappa={kappa:.4f}')

    # Summary
    print(f'\n--- {model_name} Average Across {N_SPLITS} Folds ---')
    print(f'  Accuracy  : {np.mean(acc_list):.4f} ± {np.std(acc_list):.4f}')
    print(f'  Precision : {np.mean(prec_list):.4f} ± {np.std(prec_list):.4f}')
    print(f'  Recall    : {np.mean(rec_list):.4f} ± {np.std(rec_list):.4f}')
    print(f'  F1-score  : {np.mean(f1_list):.4f} ± {np.std(f1_list):.4f}')
    print(f"  Cohen's κ : {np.mean(kappa_list):.4f} ± {np.std(kappa_list):.4f}")

    # Overall report on aggregated predictions
    present_labels = sorted(np.unique(all_y_true))
    target_names   = [STAGE_NAMES.get(l, str(l)) for l in present_labels]

    print(f'\n--- Overall Classification Report ({model_name}) ---')
    print(classification_report(
        all_y_true, all_y_pred,
        labels=present_labels,
        target_names=target_names,
        zero_division=0
    ))

    return {
        'name'     : model_name,
        'acc'      : np.mean(acc_list),
        'f1'       : np.mean(f1_list),
        'kappa'    : np.mean(kappa_list),
        'y_true'   : all_y_true,
        'y_pred'   : all_y_pred,
    }

print('run_cv() function defined.')

## CELL 13 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_results = run_cv(rf, X, y, kf, model_name='Random Forest')

## CELL 14 — Gradient Boosting with GridSearchCV

In [ ]:
param_grid_gbc = {
    'n_estimators' : [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth'    : [3, 5],
}

gbc_base = GradientBoostingClassifier(random_state=RANDOM_STATE)
print('Running GridSearchCV for GradientBoosting...')

# GridSearchCV uses its own CV — no data leakage because it sees X,y before splitting
# but we will then re-run the best model in run_cv() for honest hold-out estimates
grid_gbc = GridSearchCV(
    gbc_base, param_grid_gbc,
    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_weighted', n_jobs=-1, verbose=0
)
grid_gbc.fit(X, y)
print(f'Best GBC params : {grid_gbc.best_params_}')
print(f'Best CV F1      : {grid_gbc.best_score_:.4f}')

best_gbc = GradientBoostingClassifier(
    **grid_gbc.best_params_, random_state=RANDOM_STATE
)
gbc_results = run_cv(best_gbc, X, y, kf, model_name='Gradient Boosting')

## CELL 15 — XGBoost
**FIX 10:** `use_label_encoder` removed from parameter grid (deprecated in XGBoost ≥1.6).

In [ ]:
# FIX 10: 'use_label_encoder' removed — deprecated and produces UserWarning in all fits
param_grid_xgb = {
    'n_estimators' : [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth'    : [3, 5],
}

n_classes = len(np.unique(y))
xgb_base = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric='mlogloss',
    verbosity=0
)

print('Running GridSearchCV for XGBoost...')
grid_xgb = GridSearchCV(
    xgb_base, param_grid_xgb,
    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_weighted', n_jobs=-1, verbose=0
)
grid_xgb.fit(X, y)
print(f'Best XGB params : {grid_xgb.best_params_}')
print(f'Best CV F1      : {grid_xgb.best_score_:.4f}')

best_xgb = XGBClassifier(
    **grid_xgb.best_params_,
    random_state=RANDOM_STATE,
    eval_metric='mlogloss',
    verbosity=0
)
xgb_results = run_cv(best_xgb, X, y, kf, model_name='XGBoost')

## CELL 16 — LightGBM

In [ ]:
param_grid_lgbm = {
    'n_estimators' : [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth'    : [3, 5],
}

lgbm_base = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)

print('Running GridSearchCV for LightGBM...')
grid_lgbm = GridSearchCV(
    lgbm_base, param_grid_lgbm,
    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_weighted', n_jobs=-1, verbose=0
)
grid_lgbm.fit(X, y)
print(f'Best LGBM params: {grid_lgbm.best_params_}')
print(f'Best CV F1      : {grid_lgbm.best_score_:.4f}')

best_lgbm = LGBMClassifier(
    **grid_lgbm.best_params_,
    random_state=RANDOM_STATE,
    verbose=-1
)
lgbm_results = run_cv(best_lgbm, X, y, kf, model_name='LightGBM')

## CELL 17 — Compare All Models

In [ ]:
all_results = [rf_results, gbc_results, xgb_results, lgbm_results]

comparison_df = pd.DataFrame([
    {'Model': r['name'], 'Accuracy': r['acc'], 'F1-score': r['f1'], "Cohen's Kappa": r['kappa']}
    for r in all_results
]).set_index('Model').round(4)

print('=== MODEL COMPARISON (honest CV — SMOTE inside fold) ===')
print(comparison_df.to_string())

best_name = comparison_df['F1-score'].idxmax()
print(f'\nBest model by F1-score: {best_name}')

# Bar chart
comparison_df[['Accuracy','F1-score']].plot(
    kind='bar', figsize=(9,5), colormap='Set2', edgecolor='black'
)
plt.title('Model Comparison (Honest CV — SMOTE inside fold)')
plt.ylabel('Score'); plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

## CELL 18 — Confusion Matrix for Best Model

In [ ]:
# Get best model results
best_result = next(r for r in all_results if r['name'] == best_name)
y_true = np.array(best_result['y_true'])
y_pred = np.array(best_result['y_pred'])

present_labels = sorted(np.unique(y_true))
tick_labels    = [STAGE_NAMES.get(l, str(l)) for l in present_labels]

cm = confusion_matrix(y_true, y_pred, labels=present_labels)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=tick_labels, yticklabels=tick_labels
)
plt.title(f'Confusion Matrix — {best_name}')
plt.xlabel('Predicted Stage')
plt.ylabel('True Stage')
plt.tight_layout()
plt.savefig('confusion_matrix_best.png', dpi=150)
plt.show()
print(f'Confusion matrix saved for {best_name}.')

## CELL 19 — Final Summary

In [ ]:
print('=' * 55)
print('FINAL SUMMARY')
print('=' * 55)

print(f'\nDataset       : {os.path.basename(raw_data_objects[-1].filenames[0])}')
print(f'Total samples : {X.shape[0]} epochs x {X.shape[1]} features')

print('\nClass distribution:')
for lbl, cnt in zip(*np.unique(y, return_counts=True)):
    print(f'  {STAGE_NAMES.get(lbl,"?")} (class {lbl}): {cnt} samples')

print('\nModel performance (honest CV with SMOTE inside fold):')
print(comparison_df.to_string())

print(f'\nBest model: {best_name}')
print(f'  Accuracy  : {best_result["acc"]:.4f}')
print(f'  F1-score  : {best_result["f1"]:.4f}')
print(f"  Cohen's κ : {best_result['kappa']:.4f}")

print()
print('NOTE: Only sub-1 PSG data used. For a generalisable model')
print('add more subjects and use leave-one-subject-out CV.')